In [7]:
import os
import numpy as np
import pandas as pd


def generate_ewma_cov(
    lam=0.94,
    h_init=50,
    save_from="2021-01-01",
    in_path="../data/log_returns.csv",
    out_dir="../models",
    prefix="ewma_cov",
    debug=False
):
    """
    Generate EWMA covariance forecasts.

    Convention:
    Forecast dated t is computed using returns up to t-1.

    Parameters
    ----------
    lam : float
        EWMA decay factor.
    h_init : int
        Number of initial observations used to form starting covariance matrix.
    save_from : str
        First date to save forecasts from.
    in_path : str
        Path to log_returns.csv.
    out_dir : str
        Output directory.
    prefix : str
        Filename prefix.
    debug : bool
        If True, print sanity checks.

    Returns
    -------
    ewma_df : pd.DataFrame
        DataFrame with one row per date and full NxN covariance matrix flattened.
    """

    # -----------------------
    # Load returns
    # -----------------------
    rets = pd.read_csv(in_path, parse_dates=["Date"]).set_index("Date").sort_index()
    assets = rets.columns.tolist()
    R = rets.to_numpy()
    dates = rets.index
    T, N = R.shape

    if T <= h_init + 2:
        raise ValueError("Not enough data for the chosen h_init.")

    # -----------------------
    # Initial covariance
    # -----------------------
    Sigma = np.cov(R[:h_init].T, bias=False)

    # -----------------------
    # Full matrix column names
    # -----------------------
    colnames = [
        f"cov_{assets[i]}__{assets[j]}"
        for i in range(N)
        for j in range(N)
    ]

    # -----------------------
    # Run EWMA recursion
    # store Sigma_t on date t using r_{t-1}
    # -----------------------
    rows = []
    out_dates = []

    for k in range(h_init, T):
        r_prev = R[k - 1].reshape(N, 1)   # r_{t-1}
        Sigma = lam * Sigma + (1 - lam) * (r_prev @ r_prev.T)

        # enforce symmetry
        Sigma = 0.5 * (Sigma + Sigma.T)

        rows.append(Sigma.flatten())
        out_dates.append(dates[k])

    ewma_df = pd.DataFrame(rows, index=out_dates, columns=colnames)
    ewma_df.index.name = "Date"

    # -----------------------
    # Keep only from save_from
    # -----------------------
    ewma_df = ewma_df.loc[pd.to_datetime(save_from):]

    # -----------------------
    # Save
    # -----------------------
    os.makedirs(out_dir, exist_ok=True)

    lam_str = str(lam).replace(".", "")
    out_path = os.path.join(out_dir, f"{prefix}_lambda{lam_str}.csv")
    ewma_df.to_csv(out_path)

    print("Saved:", out_path)
    print("EWMA shape:", ewma_df.shape)
    print("EWMA date range:", ewma_df.index.min().date(), "->", ewma_df.index.max().date())

    # -----------------------
    # Optional sanity check
    # -----------------------
    if debug:
        first_date = ewma_df.index[0]
        first_loc = dates.get_loc(first_date)

        print("\nSanity check:")
        print("forecast date t:", first_date.date())
        print("uses return from t-1:", dates[first_loc - 1].date())

        # manual reconstruction of first saved EWMA matrix
        Sigma_manual = np.cov(R[:h_init].T, bias=False)
        for k in range(h_init, first_loc + 1):
            r_prev = R[k - 1].reshape(N, 1)
            Sigma_manual = lam * Sigma_manual + (1 - lam) * (r_prev @ r_prev.T)
            Sigma_manual = 0.5 * (Sigma_manual + Sigma_manual.T)

        stored = ewma_df.iloc[0].to_numpy().reshape(N, N)
        diff = np.abs(Sigma_manual - stored).max()
        print("max difference manual vs stored:", diff)

    return ewma_df

In [8]:
import os
import numpy as np
import pandas as pd


def generate_ewma_cov(
    lam=0.94,
    h_init=50,
    save_from="2021-01-01",
    in_path="../data/log_returns.csv",
    out_dir="../models",
    prefix="ewma_cov",
    debug=False
):
    """
    Generate EWMA covariance forecasts.

    Convention:
    Forecast dated t is computed using returns up to t-1.

    Parameters
    ----------
    lam : float
        EWMA decay factor.
    h_init : int
        Number of initial observations used to form starting covariance matrix.
    save_from : str
        First date to save forecasts from.
    in_path : str
        Path to log_returns.csv.
    out_dir : str
        Output directory.
    prefix : str
        Filename prefix.
    debug : bool
        If True, print sanity checks.

    Returns
    -------
    ewma_df : pd.DataFrame
        DataFrame with one row per date and full NxN covariance matrix flattened.
    """

    # -----------------------
    # Load returns
    # -----------------------
    rets = pd.read_csv(in_path, parse_dates=["Date"]).set_index("Date").sort_index()
    assets = rets.columns.tolist()
    R = rets.to_numpy()
    dates = rets.index
    T, N = R.shape

    if T <= h_init + 2:
        raise ValueError("Not enough data for the chosen h_init.")

    # -----------------------
    # Initial covariance
    # -----------------------
    Sigma = np.cov(R[:h_init].T, bias=False)

    # -----------------------
    # Full matrix column names
    # -----------------------
    colnames = [
        f"cov_{assets[i]}__{assets[j]}"
        for i in range(N)
        for j in range(N)
    ]

    # -----------------------
    # Run EWMA recursion
    # store Sigma_t on date t using r_{t-1}
    # -----------------------
    rows = []
    out_dates = []

    for k in range(h_init, T):
        r_prev = R[k - 1].reshape(N, 1)   # r_{t-1}
        Sigma = lam * Sigma + (1 - lam) * (r_prev @ r_prev.T)

        # enforce symmetry
        Sigma = 0.5 * (Sigma + Sigma.T)

        rows.append(Sigma.flatten())
        out_dates.append(dates[k])

    ewma_df = pd.DataFrame(rows, index=out_dates, columns=colnames)
    ewma_df.index.name = "Date"

    # -----------------------
    # Keep only from save_from
    # -----------------------
    ewma_df = ewma_df.loc[pd.to_datetime(save_from):]

    # -----------------------
    # Save
    # -----------------------
    os.makedirs(out_dir, exist_ok=True)

    lam_str = str(lam).replace(".", "")
    out_path = os.path.join(out_dir, f"{prefix}_lambda{lam_str}.csv")
    ewma_df.to_csv(out_path)

    print("Saved:", out_path)
    print("EWMA shape:", ewma_df.shape)
    print("EWMA date range:", ewma_df.index.min().date(), "->", ewma_df.index.max().date())

    # -----------------------
    # Optional sanity check
    # -----------------------
    if debug:
        first_date = ewma_df.index[0]
        first_loc = dates.get_loc(first_date)

        print("\nSanity check:")
        print("forecast date t:", first_date.date())
        print("uses return from t-1:", dates[first_loc - 1].date())

        # manual reconstruction of first saved EWMA matrix
        Sigma_manual = np.cov(R[:h_init].T, bias=False)
        for k in range(h_init, first_loc + 1):
            r_prev = R[k - 1].reshape(N, 1)
            Sigma_manual = lam * Sigma_manual + (1 - lam) * (r_prev @ r_prev.T)
            Sigma_manual = 0.5 * (Sigma_manual + Sigma_manual.T)

        stored = ewma_df.iloc[0].to_numpy().reshape(N, N)
        diff = np.abs(Sigma_manual - stored).max()
        print("max difference manual vs stored:", diff)

    return ewma_df

In [9]:
ewma_094 = generate_ewma_cov(lam=0.94, debug=True)
ewma_094.head()

Saved: ../models/ewma_cov_lambda094.csv
EWMA shape: (1255, 64)
EWMA date range: 2021-01-04 -> 2025-12-31

Sanity check:
forecast date t: 2021-01-04
uses return from t-1: 2020-12-31
max difference manual vs stored: 0.0


,cov_US_EQUITY__US_EQUITY,cov_US_EQUITY__INTL_EQUITY,cov_US_EQUITY__US_BONDS,cov_US_EQUITY__INTL_BONDS,cov_US_EQUITY__US_REIT,cov_US_EQUITY__INTL_REIT,cov_US_EQUITY__GOLD,cov_US_EQUITY__BTC,cov_INTL_EQUITY__US_EQUITY,cov_INTL_EQUITY__INTL_EQUITY,...,cov_GOLD__GOLD,cov_GOLD__BTC,cov_BTC__US_EQUITY,cov_BTC__INTL_EQUITY,cov_BTC__US_BONDS,cov_BTC__INTL_BONDS,cov_BTC__US_REIT,cov_BTC__INTL_REIT,cov_BTC__GOLD,cov_BTC__BTC
Date,,,,,,,,,,,,,,,,,,,,,
2021-01-04,0.000052,0.000042,-1.013893e-07,-0.000002,0.000053,0.000037,0.000017,0.000065,0.000042,0.000074,...,0.000075,0.000044,0.000065,0.000012,1.248792e-05,-0.000006,0.000087,4.921141e-05,0.000044,0.002489
2021-01-05,0.000060,0.000036,9.614280e-07,-0.000001,0.000079,0.000043,-0.000003,-0.000020,0.000036,0.000071,...,0.000102,0.000176,-0.000020,0.000038,4.309048e-06,-0.000008,-0.000121,-9.993554e-06,0.000176,0.002910
2021-01-06,0.000059,0.000038,5.012060e-07,-0.000002,0.000074,0.000047,-0.000002,0.000005,0.000038,0.000076,...,0.000096,0.000173,0.000005,0.000080,3.106012e-07,-0.000013,-0.000114,5.292666e-05,0.000173,0.002961
2021-01-07,0.000058,0.000040,-1.309565e-06,-0.000002,0.000069,0.000044,-0.000007,0.000034,0.000040,0.000078,...,0.000105,0.000087,0.000034,0.000125,-2.338019e-05,-0.000017,-0.000110,4.975106e-05,0.000087,0.003167
2021-01-08,0.000068,0.000039,-2.133485e-06,-0.000002,0.000065,0.000031,-0.000010,0.000091,0.000039,0.000073,...,0.000100,0.000069,0.000091,0.000125,-2.608438e-05,-0.000017,-0.000103,-4.206621e-07,0.000069,0.003246


In [10]:
proxy = pd.read_csv("../proxy/realized_cov_h21.csv", parse_dates=["Date"]).set_index("Date")
ewma  = pd.read_csv("../models/ewma_cov_lambda094.csv", parse_dates=["Date"]).set_index("Date")

common = proxy.index.intersection(ewma.index)

print("Common dates:", len(common), common.min().date(), "->", common.max().date())

Common dates: 1235 2021-01-04 -> 2025-12-02
